In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
agri_companies_df_dir = "input/farm_data/num_farms_province_2013.csv" 
agri_employees_df_dir = "input/farm_data/num_agriculture_workers_province_2013.csv"
preprocessed_farm_shp_dir = "output/preprocessed_farm.shp"
gemeente_shp_dir = "input/gemeentes/corrected/Gemeentes2013TrMr.shp"

preprocessed_farm_shp_new_dir = "output/preprocessed_farm_1.shp"

#### Preprocessing and joining two tables

In [3]:
df_agri_companies = pd.read_csv(agri_companies_df_dir, delimiter=";")
df_agri_employees = pd.read_csv(agri_employees_df_dir, delimiter=";")

In [4]:
df_agri_employees.head(5)

,Geslacht,Perioden,Regio's,"Arbeidskrachten/Regelmatig werkzaam/Regelmatig werkzaam, totaal (aantal)",Arbeidskrachten/Niet-regelmatig werkzaam (aantal)
0,Totaal,2013,Nederland,193244,NaN
1,Totaal,2013,Groningen (PV),8191,NaN
2,Totaal,2013,Fryslân (PV),13615,NaN
3,Totaal,2013,Drenthe (PV),8889,NaN
4,Totaal,2013,Overijssel (PV),20339,NaN


In [5]:
df_agri_companies = df_agri_companies.rename(columns={
  "Regio's": "Provinces",
  "Aantal landbouwbedrijven, totaal (aantal)": "Tot. agricultural companies",
  "Graasdieren/Aantal bedrijven/Graasdieren, totaal (aantal)": "Tot. grass animal companies",
  "Hokdieren/Aantal bedrijven/Hokdieren, totaal (aantal)": "Tot. caged animal companies"
})

# Summing the animal companies to a new column
df_agri_companies["Tot. animal companies"] = (df_agri_companies["Tot. grass animal companies"] 
                                               + df_agri_companies["Tot. caged animal companies"])

# Dropping the individual animal company types 
df_agri_companies = df_agri_companies.drop(columns={"Tot. grass animal companies", "Tot. caged animal companies"})

df_agri_companies.head(5)

,Perioden,Provinces,Tot. agricultural companies,Tot. animal companies
0,2013,Nederland,67481,50181
1,2013,Groningen (PV),3215,2264
2,2013,Fryslân (PV),5501,5076
3,2013,Drenthe (PV),3579,2776
4,2013,Overijssel (PV),8283,7975


In [6]:
df_agri_employees = df_agri_employees.rename(columns={
  "Regio's": "Provinces",
  "Arbeidskrachten/Regelmatig werkzaam/Regelmatig werkzaam, totaal (aantal)": "Tot. regular working employees",
  "Arbeidskrachten/Niet-regelmatig werkzaam (aantal)": "Tot. non-regular working employees"
})

df_agri_employees.head(5)

# Dropping column non-regular working employees since it is all NaN
df_agri_employees = df_agri_employees.drop(["Tot. non-regular working employees", "Geslacht"], axis=1)

In [7]:
# Joining the two tables based on the regions.
df_agri = pd.merge(df_agri_employees, df_agri_companies, on=["Provinces", "Perioden"], how="inner")
df_agri.head(20)

,Perioden,Provinces,Tot. regular working employees,Tot. agricultural companies,Tot. animal companies
0,2013,Nederland,193244,67481,50181
1,2013,Groningen (PV),8191,3215,2264
2,2013,Fryslân (PV),13615,5501,5076
3,2013,Drenthe (PV),8889,3579,2776
4,2013,Overijssel (PV),20339,8283,7975
5,2013,Flevoland (PV),5687,1806,604
6,2013,Gelderland (PV),28791,11516,10186
7,2013,Utrecht (PV),7109,2779,2632
8,2013,Noord-Holland (PV),17544,4588,2527
9,2013,Zuid-Holland (PV),27056,6499,3243


#### Determining the number of farm workers per region in NL

In [8]:
df_agri["Num. employees per agricultural company"] = df_agri["Tot. regular working employees"]/df_agri["Tot. agricultural companies"]
df_agri.head(5)

,Perioden,Provinces,Tot. regular working employees,Tot. agricultural companies,Tot. animal companies,Num. employees per agricultural company
0,2013,Nederland,193244,67481,50181,2.863680
1,2013,Groningen (PV),8191,3215,2264,2.547745
2,2013,Fryslân (PV),13615,5501,5076,2.475005
3,2013,Drenthe (PV),8889,3579,2776,2.483655
4,2013,Overijssel (PV),20339,8283,7975,2.455511


In [9]:
df_agri["Tot. employees animal companies"] = df_agri["Num. employees per agricultural company"] *  df_agri["Tot. animal companies"]
df_agri

,Perioden,Provinces,Tot. regular working employees,Tot. agricultural companies,Tot. animal companies,Num. employees per agricultural company,Tot. employees animal companies
0,2013,Nederland,193244,67481,50181,2.863680,143702.333457
1,2013,Groningen (PV),8191,3215,2264,2.547745,5768.094557
2,2013,Fryslân (PV),13615,5501,5076,2.475005,12563.123069
3,2013,Drenthe (PV),8889,3579,2776,2.483655,6894.625314
4,2013,Overijssel (PV),20339,8283,7975,2.455511,19582.702523
5,2013,Flevoland (PV),5687,1806,604,3.148948,1901.964563
6,2013,Gelderland (PV),28791,11516,10186,2.500087,25465.884509
7,2013,Utrecht (PV),7109,2779,2632,2.558114,6732.957179
8,2013,Noord-Holland (PV),17544,4588,2527,3.823888,9662.965998
9,2013,Zuid-Holland (PV),27056,6499,3243,4.163102,13500.939837


In [10]:
df_agri["Provinces"] = df_agri["Provinces"].str.replace(" (PV)", "", regex=False)
df_agri["Provinces"] = df_agri["Provinces"].replace({
    "Fryslân": "Friesland"
})
df_agri

,Perioden,Provinces,Tot. regular working employees,Tot. agricultural companies,Tot. animal companies,Num. employees per agricultural company,Tot. employees animal companies
0,2013,Nederland,193244,67481,50181,2.863680,143702.333457
1,2013,Groningen,8191,3215,2264,2.547745,5768.094557
2,2013,Friesland,13615,5501,5076,2.475005,12563.123069
3,2013,Drenthe,8889,3579,2776,2.483655,6894.625314
4,2013,Overijssel,20339,8283,7975,2.455511,19582.702523
5,2013,Flevoland,5687,1806,604,3.148948,1901.964563
6,2013,Gelderland,28791,11516,10186,2.500087,25465.884509
7,2013,Utrecht,7109,2779,2632,2.558114,6732.957179
8,2013,Noord-Holland,17544,4588,2527,3.823888,9662.965998
9,2013,Zuid-Holland,27056,6499,3243,4.163102,13500.939837


In [11]:
# Rounding the number of 
df_agri["Tot. employees animal companies"] = df_agri["Tot. employees animal companies"].round()
df_agri

,Perioden,Provinces,Tot. regular working employees,Tot. agricultural companies,Tot. animal companies,Num. employees per agricultural company,Tot. employees animal companies
0,2013,Nederland,193244,67481,50181,2.863680,143702.0
1,2013,Groningen,8191,3215,2264,2.547745,5768.0
2,2013,Friesland,13615,5501,5076,2.475005,12563.0
3,2013,Drenthe,8889,3579,2776,2.483655,6895.0
4,2013,Overijssel,20339,8283,7975,2.455511,19583.0
5,2013,Flevoland,5687,1806,604,3.148948,1902.0
6,2013,Gelderland,28791,11516,10186,2.500087,25466.0
7,2013,Utrecht,7109,2779,2632,2.558114,6733.0
8,2013,Noord-Holland,17544,4588,2527,3.823888,9663.0
9,2013,Zuid-Holland,27056,6499,3243,4.163102,13501.0


#### Adding number of farm employees for each farm

In [12]:
preprocessed_farm_gdf = gpd.read_file(preprocessed_farm_shp_dir)
preprocessed_farm_gdf.head(5)


,node,gm_id,geometry
0,395,369,POINT (1392035.976 565963.947)
1,396,369,POINT (1391399.655 569395.729)
2,417,369,POINT (1393344.756 564431.652)
3,418,369,POINT (1392708.534 567863.542)
4,419,369,POINT (1392072.069 571295.356)


In [13]:
# For each of Boris' points, we determine which province they belong to and attach the number
# farm employees accordingly
gemeente_gdf = gpd.read_file(gemeente_shp_dir)
gemeente_gdf.head(5)

,OBJECTID,gemeente,cbs_code,provincie,prov_id,x,y,Shape_Leng,Shape_Area,geomX,geomY,geometry
0,1,Appingedam,0003,Groningen,20,252075.0,593481.0,0.311202,0.003314,6.84957,53.3171,"POLYGON ((1588286.399 834196.416, 1588290.233 ..."
1,2,Bedum,0005,Groningen,20,236060.0,590896.0,0.404303,0.006058,6.59716,53.2948,"POLYGON ((1567602.439 825774.202, 1567591.997 ..."
2,3,Bellingwedde,0007,Groningen,20,271170.0,569542.0,0.583319,0.014766,7.12118,53.0986,"POLYGON ((1615256.216 811055.136, 1615284.41 8..."
3,4,Ten Boer,0009,Groningen,20,241817.0,588286.0,0.398156,0.006159,6.69145,53.2788,"POLYGON ((1573481.825 829474.947, 1573484.831 ..."
4,5,Delfzijl,0010,Groningen,20,256184.0,594735.0,1.222306,0.018322,6.92970,53.3200,"POLYGON ((1588290.233 834174.07, 1588286.399 8..."


In [14]:
gemeente_gdf = gemeente_gdf[["OBJECTID", "provincie"]]
gemeente_gdf = gemeente_gdf.rename(columns={
  "OBJECTID": "gm_id",
  "provincie": "Provinces"
  })

gemeente_gdf.head(5)

,gm_id,Provinces
0,1,Groningen
1,2,Groningen
2,3,Groningen
3,4,Groningen
4,5,Groningen


In [15]:
preprocessed_farm_gdf_new = preprocessed_farm_gdf.merge(gemeente_gdf, on="gm_id", how="inner")
preprocessed_farm_gdf_new.head(5)

,node,gm_id,geometry,Provinces
0,395,369,POINT (1392035.976 565963.947),Zeeland
1,396,369,POINT (1391399.655 569395.729),Zeeland
2,417,369,POINT (1393344.756 564431.652),Zeeland
3,418,369,POINT (1392708.534 567863.542),Zeeland
4,419,369,POINT (1392072.069 571295.356),Zeeland


In [16]:
preprocessed_farm_gdf_new = preprocessed_farm_gdf_new.merge(df_agri[["Provinces", "Tot. employees animal companies"]], on="Provinces", how="inner")
preprocessed_farm_gdf_new.head(10)

,node,gm_id,geometry,Provinces,Tot. employees animal companies
0,395,369,POINT (1392035.976 565963.947),Zeeland,2897.0
1,396,369,POINT (1391399.655 569395.729),Zeeland,2897.0
2,417,369,POINT (1393344.756 564431.652),Zeeland,2897.0
3,418,369,POINT (1392708.534 567863.542),Zeeland,2897.0
4,419,369,POINT (1392072.069 571295.356),Zeeland,2897.0
5,420,369,POINT (1391435.362 574727.095),Zeeland,2897.0
6,440,369,POINT (1394017.394 566331.272),Zeeland,2897.0
7,441,369,POINT (1393381.029 569763.194),Zeeland,2897.0
8,442,369,POINT (1392744.421 573195.041),Zeeland,2897.0
9,463,369,POINT (1395326.236 564798.919),Zeeland,2897.0


In [17]:
province_counts = preprocessed_farm_gdf_new["Provinces"].value_counts()
print(province_counts)

Provinces
Gelderland       1476
Noord-Brabant    1457
Friesland        1021
Overijssel        987
Zuid-Holland      864
Noord-Holland     821
Drenthe           777
Groningen         691
Limburg           638
Zeeland           530
Flevoland         425
Utrecht           418
Name: count, dtype: int64


In [18]:
list_1 = ['Drenthe', 'Fryslan', 'Amsterdam', 'Noord- en Oost-Gelderland',
       'Zuid-Holland Zuid', 'Rotterdam-Rijnmond', 'Hollands Noorden',
       'Twente', 'Flevoland', 'Hollands Midden', 'Utrecht', 'Groningen',
       'Gelderland-Midden', 'Brabant-Zuidoost', 'Zaanstreek-Waterland',
       'Limburg-Noord', 'Gelderland-Zuid', 'Kennemerland',
       'Gooi & Vechtstreek', 'Zeeland', 'Ijsselland']

list_new = list(preprocessed_farm_gdf_new["Provinces"].unique())

In [19]:
len(list_new)

12

In [20]:
for province in list_1:
  if province not in list_new:
    print(province)

Fryslan
Amsterdam
Noord- en Oost-Gelderland
Zuid-Holland Zuid
Rotterdam-Rijnmond
Hollands Noorden
Twente
Hollands Midden
Gelderland-Midden
Brabant-Zuidoost
Zaanstreek-Waterland
Limburg-Noord
Gelderland-Zuid
Kennemerland
Gooi & Vechtstreek
Ijsselland


In [21]:
preprocessed_farm_gdf_new = preprocessed_farm_gdf_new.rename(columns={"Tot. employees animal companies": "no_farmers"})
preprocessed_farm_gdf_new["no_farmers"] = preprocessed_farm_gdf_new["no_farmers"].astype(int)

preprocessed_farm_gdf_new = preprocessed_farm_gdf_new.drop("Provinces", axis =1)

preprocessed_farm_gdf_new.to_file(preprocessed_farm_shp_new_dir)

In [22]:
preprocessed_farm_gdf_new.head(5)

,node,gm_id,geometry,no_farmers
0,395,369,POINT (1392035.976 565963.947),2897
1,396,369,POINT (1391399.655 569395.729),2897
2,417,369,POINT (1393344.756 564431.652),2897
3,418,369,POINT (1392708.534 567863.542),2897
4,419,369,POINT (1392072.069 571295.356),2897


### Determining the number of farmers per GGD region

In [23]:
df_ggd_dir = "../imbit_model/data/input/spatial_layers/GGDpointsTrMr.shp"
df_ggd = gpd.read_file(df_ggd_dir)

df_provincie_dir = "../../GoHot/data/cbsgebiedsindelingen2016-2025/cbsgebiedsindelingen2018.gpkg"
df_provincie = gpd.read_file(df_provincie_dir, layer = "provincie_gegeneraliseerd")
df_provincie

,id,statcode,jrstatcode,statnaam,rubriek,geometry
0,1,PV20,2018PV20,Groningen,provincie,"MULTIPOLYGON (((247426.661 609175.775, 245943...."
1,2,PV21,2018PV21,Friesland,provincie,"MULTIPOLYGON (((208007.047 603377.135, 207751...."
2,3,PV22,2018PV22,Drenthe,provincie,"MULTIPOLYGON (((228930.078 579661.963, 228872...."
3,4,PV23,2018PV23,Overijssel,provincie,"MULTIPOLYGON (((205510.891 539331.068, 205075...."
4,5,PV24,2018PV24,Flevoland,provincie,"MULTIPOLYGON (((179365.35 539224.91, 179156.93..."
5,6,PV25,2018PV25,Gelderland,provincie,"MULTIPOLYGON (((191251.266 499559.5, 190471.82..."
6,7,PV26,2018PV26,Utrecht,provincie,"MULTIPOLYGON (((130059 479450, 129536.608 4795..."
7,8,PV27,2018PV27,Noord-Holland,provincie,"MULTIPOLYGON (((132658.065 550437.253, 140200...."
8,9,PV28,2018PV28,Zuid-Holland,provincie,"MULTIPOLYGON (((97460.781 481116.307, 94175.33..."
9,10,PV29,2018PV29,Zeeland,provincie,"MULTIPOLYGON (((48350.053 419755.278, 48127.94..."


In [24]:
# Determining which province each GGDregion belongs to

dict_provincie = {}

df_provincie = df_provincie.to_crs(df_ggd.crs)

# Loop through each province row
for idx, prov in df_provincie.iterrows():
    prov_name = prov["statnaam"]  # adjust to your actual column name
    prov_geom = prov.geometry

    # Select GGD points that fall within this province
    ggd_in_province = df_ggd[df_ggd.within(prov_geom)]

    # Add to dictionary
    dict_provincie[prov_name] = ggd_in_province["GGDREGIO"].tolist()  # adjust col name if needed

# Inspect result
for k, v in dict_provincie.items():
    print(k, ":", v)

Groningen : ['Groningen']
Friesland : ['Fryslan']
Drenthe : ['Drenthe']
Overijssel : ['Ijsselland', 'Twente']
Flevoland : ['Flevoland']
Gelderland : ['Noord- en Oost-Gelderland', 'Gelderland-Midden', 'Gelderland-Zuid']
Utrecht : ['Utrecht']
Noord-Holland : ['Hollands Noorden', 'Zaanstreek-Waterland', 'Amsterdam', 'Kennemerland', 'Gooi & Vechtstreek']
Zuid-Holland : ['Haaglanden', 'Hollands Midden', 'Rotterdam-Rijnmond', 'Zuid-Holland Zuid']
Zeeland : ['Zeeland']
Noord-Brabant : ['Hart voor Brabant', 'West-Brabant', 'Brabant-Zuidoost']
Limburg : ['Limburg-Noord', 'Zuid-Limburg']


For each province, we have the GGDregions located in the province and the total number of farm employees. 
For each farm we set the number of farmers = Num. employees per agricultural company for the Province that the farm falls in.

In [25]:
dict_ggd_province = {}
for prov, ggd_list in dict_provincie.items():
    for ggd in ggd_list:
        dict_ggd_province[ggd] = prov

In [26]:
def compute_num_farmers(row):
    province = row["PROVINCE"]
    # get total employees for this province
    total_employees = df_agri.loc[df_agri["Provinces"] == province, "Tot. employees animal companies"].values[0]
    # divide equally among GGD regions in that province
    return total_employees / len(dict_provincie[province])

In [27]:
# For each GGD region we set the number of farmers = total farmers 
df_centroid_farmers = pd.DataFrame()

df_centroid_farmers["GGDREGION"] = df_ggd["GGDREGIO"].tolist()
df_centroid_farmers["PROVINCE"] = df_centroid_farmers["GGDREGION"].map(dict_ggd_province)
df_centroid_farmers["num_farmers_ggdregio"] = df_centroid_farmers.apply(compute_num_farmers, axis=1)

df_centroid_farmers

,GGDREGION,PROVINCE,num_farmers_ggdregio
0,Groningen,Groningen,5768.000000
1,Fryslan,Friesland,12563.000000
2,Drenthe,Drenthe,6895.000000
3,Hollands Noorden,Noord-Holland,1932.600000
4,Flevoland,Flevoland,1902.000000
5,Ijsselland,Overijssel,9791.500000
6,Zaanstreek-Waterland,Noord-Holland,1932.600000
7,Amsterdam,Noord-Holland,1932.600000
8,Kennemerland,Noord-Holland,1932.600000
9,Gooi & Vechtstreek,Noord-Holland,1932.600000


In [28]:
file_dir = "../imbit_model/data/input/imm_startpop/StartPopulationHighestR90TrMr.shp"
df = gpd.read_file(file_dir)

df = df.merge(
    df_centroid_farmers[["GGDREGION", "num_farmers_ggdregio"]],
    on="GGDREGION",
    how="left"
)

In [32]:
df[df["GGDREGION"]=="Utrecht"]

,ID,Name,GGDREGION,DKTPVACCIN,SumOutJob,SumInJob,Difference,SumOutScho,SumInSchoo,F_OutComm,...,S18112,E18112,I18112,R18112,S19111,E19111,I19111,R19111,geometry,num_farmers_ggdregio
12,13,Amersfoort,Utrecht,98.5,72300,80200,7900,1,1,48.31,...,146.57,0.0,0.0,1319.10,2518.81,0.0,0.0,22669.30,POINT (1513380.146 687379.078),6733.0
20,21,Baarn,Utrecht,96.9,9000,1600,-7400,1,1,37.07,...,25.54,0.0,0.0,229.82,408.58,0.0,0.0,3677.24,POINT (1504573.518 689172.545),6733.0
57,58,Bunnik,Utrecht,97.2,5800,5500,-300,1,1,39.83,...,15.06,0.0,0.0,135.53,245.10,0.0,0.0,2205.86,POINT (1504704.337 670578.893),6733.0
58,59,Bunschoten,Utrecht,96.9,9000,8200,-800,1,1,44.30,...,20.42,0.0,0.0,183.79,341.92,0.0,0.0,3077.26,POINT (1510214.805 694217.486),6733.0
70,71,De0Bilt,Utrecht,96.4,16100,14100,-2000,1,1,38.30,...,43.88,0.0,0.0,394.90,707.40,0.0,0.0,6366.59,POINT (1499744.047 681179.547),6733.0
72,73,De0Ronde0Venen,Utrecht,99.1,18000,10800,-7200,1,1,42.01,...,43.70,0.0,0.0,393.32,721.10,0.0,0.0,6489.88,POINT (1480578.407 687332.553),6733.0
94,95,Eemnes,Utrecht,96.3,3300,2200,-1100,1,1,37.52,...,9.23,0.0,0.0,83.03,148.02,0.0,0.0,1332.18,POINT (1504678.175 694461.632),6733.0
163,164,Houten,Utrecht,96.3,23400,18900,-4500,1,1,48.32,...,47.42,0.0,0.0,426.82,815.06,0.0,0.0,7335.54,POINT (1503055.309 666068.902),6733.0
166,167,IJsselstein,Utrecht,99.1,15300,7800,-7500,1,1,44.67,...,34.35,0.0,0.0,309.16,576.49,0.0,0.0,5188.45,POINT (1492150.247 666763.532),6733.0
193,194,Leusden,Utrecht,100.0,12400,11900,-500,1,1,42.80,...,29.40,0.0,0.0,264.59,487.55,0.0,0.0,4387.93,POINT (1516608.458 682096.518),6733.0


In [106]:
# Adding a new column as number of farmers
# Suppose you already have total farmers per GGDREGIO in df["num_farmers_ggdregio"]

# Count how many rows per GGDREGIO
counts = df.groupby("GGDREGION")["GGDREGION"].transform("count")

# Assign S0000 = total_farmers / number of rows in that GGDREGIO
df["S00000"] = round(df["num_farmers_ggdregio"] / counts)
df["E00000"] = 0
df["I00000"] = 0
df["R00000"] = 0

In [ ]:
df = df.drop(columns="num_farmers_ggdregio")
df.to_file('../imbit_model/data/input/imm_startpop/test.shp')

In [108]:
df[["num_farmers_ggdregio"]]

,num_farmers_ggdregio
0,6895.000000
1,12563.000000
2,1932.600000
3,8488.666667
4,12563.000000
...,...
391,12563.000000
392,8488.666667
393,9791.500000
394,3375.250000


In [ ]:
counts

0      12
1      41
2      35
3      22
4      41
       ..
391    41
392    22
393    11
394    17
395    11
Name: GGDREGION, Length: 396, dtype: int64

In [ ]:
df["S00000"]

0      575.0
1      306.0
2       55.0
3      386.0
4      306.0
       ...  
391    306.0
392    386.0
393    890.0
394    199.0
395    890.0
Name: S00000, Length: 396, dtype: float64

In [ ]:
df[df["ID"]==2]["S00000"]

1    306.0
Name: S00000, dtype: float64

In [ ]:
df[["Name", "S00000"]]
# num-spillovers = (S00000 * totalinfectedanimals * prob-spillover-risk-after-contact * prob-contact

,Name,S00000
0,Aa0en0Hunze,575.0
1,Aalburg,306.0
2,Aalsmeer,55.0
3,Aalten,386.0
4,Achtkarspelen,306.0
...,...,...
391,Zundert,306.0
392,Zutphen,386.0
393,Zwartewaterland,890.0
394,Zwijndrecht,199.0
